# Homework 4: Optional Advanced Final Project (A+ Grade Bump)

Incorporate README of nationality prediction model
Debug
Ensure reproducibility

- This homework template guides you through presenting your final project analysis. 
- Use this notebook to:
    - Generate all visualizations/results and report findings with a pipeline then simply view the results here
    - Generate figures/analysis with imported scripts to produce visualizations/results, and report findings.

> **Note for Beginners:** Running modular Python scripts from inside a Jupyter notebook can sometimes lead to import path or dependency issues if the working directory changes. If you encounter import errors, make sure you add the path of your script folder to `sys.path`, or execute your pipeline directly from your terminal using:
```bash
uv run python src/final_project/first-last/pipeline.py
```

## A. Describe Project

### **Guidance**
- State the policy question, puzzle, or social science problem you are addressing.
- Frame your central hypothesis and the expected relationship between your variables.
- Describe the scope of your analysis (e.g., geographical regions, years covered).
- Highlight the datasets you selected to examine this question.

### **Project Summary**
- **Project Title:** Name-Inferred Inventor Origin and State-Level Patent Output
- **Student Name:** Lee Qi Cheng
- **Policy Relevance Statement:** Skilled-immigration policy (H-1B caps, visa restrictions) is frequently defended or opposed on the premise that foreign-born talent drives U.S. innovation. This project tests that premise directly by asking whether U.S. states with a larger name-inferred foreign-origin share of patent inventors actually produce more patents, once the size of each state's inventor pool is accounted for.
- **Central Hypothesis:** States with a higher share of foreign-origin-coded inventor names show higher patent output in 2025, controlling for the size of the state's inventor pool.
- **Scope:** Cross-sectional analysis of all 50 U.S. states plus DC, using patents granted in a single year (2025). This is not a panel or growth design. There is no year-over-year variation in the underlying data, so results describe a one-year snapshot, not a trend or a causal claim.
- **Datasets:** (1) USPTO annualized patent data (2025 grants, inventor-level), hosted on Hugging Face and pulled programmatically in Section 1. (2) A custom-trained name-to-nationality classifier (two single-layer GRU networks, one for first names and one for last names, 23 output categories), trained separately on a global first/last-name dataset and applied here to infer each inventor's likely name-coded origin.
- **Important caveat:** "Foreign-origin" here means *name-inferred* origin, that is a statistical guess based on how a name is spelled, not a measurement of citizenship, immigration status, or self-identified ethnicity. Many U.S.-born, multi-generation citizens would be coded as "foreign-origin" under this measure. Results should be read as describing name-coded patterns in patent records, not immigration status.

---

## 1. Download Data

### **Guidance**
- Run the data acquisition step by executing or importing from your `data.py` file.
- Ensure your script programmatically downloads the datasets and saves them locally.
- Verify the download by displaying the destination folder structure and confirming files are saved.

### **Data Acquisition Details**
- **Primary Data Source:** USPTO Annualized Patent Data (2025 granted patents, inventor-level), hosted on a Hugging Face dataset repo and downloaded programmatically via `huggingface_hub.hf_hub_download`.
- **Variables Retrieved:** `patent_number`, `grant_year`, `country`, `state`, `inventors` (team size), and up to 10 inventor slots per patent (`inventor_id1`-`inventor_id10`, `inventor_name1`-`inventor_name10`).
- **Local Storage Path:** `data/final_project/qicheng-lee/raw/annual_patents_raw.csv`


In [1]:
# Import and run your advanced OOP data acquisition script
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../../../src/final_project/qicheng-lee/advanced').resolve()))
import data
data_fetcher = data.DataAcquisition()
df_raw = data_fetcher.run()
df_raw.head()


Data acquired and saved to data\final_project\qicheng-lee\raw\annual_patents_raw.csv with shape (343743, 62)


,patent_number,grant_year,application_number,application_year,d_inventor,d_assignee,d_location,d_application,d_cpc,d_ipc,...,inventor_name8,male_flag8,inventor_id9,inventor_name9,male_flag9,inventor_id10,inventor_name10,male_flag10,wipo_sector_title9,wipo_field_title9
0,11694774,2025,16598793,2019,1,1,1,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12185648,2025,16972991,2019,1,1,1,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12185649,2025,17154409,2021,1,0,0,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12185650,2025,17162175,2021,1,1,1,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12185651,2025,17560959,2021,1,0,0,1,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<cell_type>markdown</cell_type>## 2. Manipulate Data

### **Guidance**
- Preprocess, clean, and merge your raw files by running your `manipulate.py` file.
- Filter out missing values, fix data types, handle outliers, and align column names.
- Merge your datasets using standardized index columns (e.g., country name, date, country-year).
- Verify the final shape of the combined dataset and display a preview (`.head()`).

### **Preprocessing Summary**
- **Filtering:** Restricted to U.S.-assigned patents (`country == "US"`) with a valid state code (50 states + DC; territories and military codes excluded).
- **Reshaping:** The 10 wide inventor slots per patent were reshaped into one row per patent-inventor pair, dropping empty slots.
- **Classification:** Each of the 244,957 unique inventors in the filtered data was classified exactly once (not once per patent) through the two trained GRU name-to-nationality models, which jointly predict one of 23 name-coded nationality categories per inventor. Classification took roughly 11 minutes.
- **Bucketing:** The 23 raw categories were collapsed into two analysis groups for the regression: (1) `domestic` (English-coded names); and (2) `foreign` (everything else, undifferentiated). No individual non-domestic nationality (e.g. Chinese, Indian) is broken out as its own regression variable. The full, uncollapsed 23-category breakdown was preserved separately for descriptive purposes only (see Figure 3).
- **Aggregation:** Collapsed to a state-level cross-section with patent counts, inventor counts, and the domestic/foreign split per state.
- **Final Dataset Observations:** 51 rows (50 states + DC) × 8 columns in the regression dataset (`state`, `patent_count`, `inventor_count`, `domestic`, `foreign`, `share_foreign`, `log_patents`, `log_inventor_count`); 23 rows × 4 columns in the separate descriptive nationality breakdown.
- **Clean Data Paths:** `data/final_project/qicheng-lee/clean/state_cross_section.csv` (regression data) and `data/final_project/qicheng-lee/clean/nationality_breakdown.csv` (descriptive breakdown).
</cell_type>

In [2]:
import manipulate
manipulator = manipulate.DataManipulation()
df_clean = manipulator.run()
df_clean.head()

Classifying inventors: 100%|██████████| 244957/244957 [11:14<00:00, 363.29it/s]


Nationality breakdown saved to data\final_project\qicheng-lee\clean\nationality_breakdown.csv with shape (23, 4)
Data processed and saved to data\final_project\qicheng-lee\clean\state_cross_section.csv with shape (51, 8)


,state,patent_count,inventor_count,domestic,foreign,share_foreign,log_patents,log_inventor_count
0,AK,7,19,10,9,0.473684,1.945910,2.944439
1,AL,292,639,306,333,0.521127,5.676754,6.459904
2,AR,307,816,243,573,0.702206,5.726848,6.704414
3,AZ,1807,3057,1124,1933,0.632319,7.499423,8.025189
4,CA,43576,77691,20620,57071,0.734590,10.682262,11.260495


## 3. Visualize Data

### **Guidance**
- Generate publication-quality visualizations by calling functions in your `graph.py` file.
- Your graphs must include proper axis labels, descriptive titles, accessible colors, and clear legends.
- Display the figures directly in this section by importing your graphing functions.

### **Visualizations & Observations**
- **Figure 1 Key Takeaway:** California and Washington have the highest name-inferred foreign-origin inventor shares (~73%), with most states clustering between roughly 45% and 65%. Some high-ranking states (e.g., ID, AR) aren't the usual tech hubs. This is likely a noisier estimate given how few inventors those states have in the sample (denominators as small as 19 in AK).
- **Figure 2 Key Takeaway:** The raw bivariate relationship looks positive, but this is driven by a state-size confound. California alone (77,691 inventors) is an extreme outlier pulling the fit upward. This figure is descriptive/pre-regression; once state size is controlled for in Section 4, the relationship is no longer statistically significant.
- **Figure 3 Key Takeaway:** The full 23-category breakdown surfaces a classifier reliability concern rather than a substantive finding: "Arabic" is predicted as the second-largest name-inferred category nationwide (~24%, behind only "English" at ~34%), while "Indian" is nearly the smallest category (~0.2-0.3%) despite Indian-origin inventors being one of the largest non-domestic groups in U.S. tech by independent measures (e.g., USCIS H-1B data). This suggests the classifier may be systematically misrouting South Asian names into the Arabic category, and tempers confidence in any Chinese/Indian-specific claims drawn from this data.


In [3]:
import graph
visualizer = graph.DataVisualization()
visualizer.run()

Figures saved to reports\final_project\qicheng-lee


[WindowsPath('C:/Users/Cheng/Documents/GitHub/datascience-publicpolicy-2026/reports/final_project/qicheng-lee/foreign_share_by_state.png'),
 WindowsPath('C:/Users/Cheng/Documents/GitHub/datascience-publicpolicy-2026/reports/final_project/qicheng-lee/patents_vs_foreign_share.png'),
 WindowsPath('C:/Users/Cheng/Documents/GitHub/datascience-publicpolicy-2026/reports/final_project/qicheng-lee/nationality_breakdown.png')]

<cell_type>markdown</cell_type>## 4. Model Data

### **Guidance**
- Run statistical modeling (such as OLS regression) using functions in your `model.py` file.
- Specify your dependent variable and independent variables clearly.
- Print out the full model summary (e.g., coefficient table, standard errors, R-squared, p-values).
- Summarize your model findings and state whether they support your initial hypothesis.

### **Model Specifications & Interpretation**
- **Model Type:** Cross-sectional Ordinary Least Squares (OLS) with heteroskedasticity-robust (HC1) standard errors, run as two specifications. A panel/fixed-effects model was not used because the data covers a single year (2025) with no year-over-year variation.
- **Dependent Variable (both models):** `log_patents` (log of state patent count)

**Model 1 — Bivariate, no control for state size:**
- **Independent Variable:** `share_foreign` only
- **Model Fit Metrics:** R-squared = 0.367, Adjusted R-squared = 0.354, N = 51
- **Coefficient of `share_foreign`:** 13.2834, Std. Err: 2.540, z: 5.230, p-value: <0.001 (95% CI: [8.31, 18.26])
- **Statistical Significance:** Strongly significant, and positive — consistent with the upward-sloping fit line visible in Figure 2.

**Model 2 — With inventor count control (primary specification):**
- **Independent Variables:** `share_foreign`, `log_inventor_count` (control for the size of each state's inventor pool)
- **Model Fit Metrics:** R-squared = 0.988, Adjusted R-squared = 0.987, N = 51
- **Coefficient of `share_foreign`:** -0.3554, Std. Err: 0.645, z: -0.551, p-value: 0.582 (95% CI: [-1.62, 0.91])
- **Coefficient of `log_inventor_count`:** 1.0306, Std. Err: 0.021, z: 48.895, p-value: <0.001
- **Statistical Significance:** `share_foreign` is no longer significant (p = 0.582), and its point estimate flips to the opposite sign from Model 1. `log_inventor_count` is highly significant and accounts for nearly all of the explained variance.

- **Key Conclusion:** Comparing the two models makes the confounding explicit rather than just implied by Figure 2. Model 1, taken alone, would suggest a strong positive relationship between foreign-origin share and patent output — but that relationship disappears entirely once state size is controlled for in Model 2, meaning it was very likely driven by the fact that larger states (more inventors) happen to also have higher foreign-origin shares, not by a genuine link between origin composition and innovation output. Taking Model 2 as the primary specification, **the central hypothesis is not supported**: after accounting for the size of a state's inventor pool, there is no statistically significant relationship between name-inferred foreign-origin share and patent output. Three limitations temper this null result rather than making it the final word: (1) the classifier reliability concern raised in Figure 3 casts doubt on what "foreign-origin" is actually capturing here; (2) with only 51 observations and one extreme outlier (California), the significant right-skew and non-normal residuals in Model 2 (Jarque-Bera p < 0.001) suggest the result may be sensitive to that single point — a California-excluded robustness check would be a natural next step; (3) this is a single-year cross-section, so the result speaks to a 2025 snapshot only, not a growth or causal claim about immigration policy's effect on innovation over time.

In [4]:
import model
modeler = model.EconometricModeling()
summary = modeler.run()

=== Model 1: Without Inventor Count Control ===
                            OLS Regression Results                            
Dep. Variable:            log_patents   R-squared:                       0.367
Model:                            OLS   Adj. R-squared:                  0.354
Method:                 Least Squares   F-statistic:                     27.35
Date:                Mon, 20 Jul 2026   Prob (F-statistic):           3.50e-06
Time:                        23:56:49   Log-Likelihood:                -88.197
No. Observations:                  51   AIC:                             180.4
Df Residuals:                      49   BIC:                             184.3
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------